In [ ]:
# !pip install  numpy matplotlib opencv-python tensorflow[and-cuda]
# !pip install  numpy matplotlib opencv-python tensorflow

import os
import json
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, BatchNormalization, LeakyReLU
import matplotlib.pyplot as plt


# base_path = "/Users/nil/Side/AIResearch/Code/sky-scan/data"
base_path = "/home/student/sky-scan/data"


In [ ]:
print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))

# Define paths
image_dir = f"{base_path}/Tiles/"  # Update with your dataset path
json_files = [f"{base_path}/Tiles/instances_1_2_3.json" ,f"{base_path}/Tiles/instances_4_5_6.json",f"{base_path}/Tiles/instances_7_8_9.json",f"{base_path}/Tiles/instances_10_11_12.json"]

annotations = []
# Load JSON annotation
for json_file in json_files:
    with open(json_file) as f:
        annotations.append(json.load(f))

import cv2
import numpy as np

def load_data(image_dir, annotations, image_size=(256, 256)):
    images, masks = [], []

    for item in annotations["images"]:
        img_path = os.path.join(image_dir, item["file_name"])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, image_size)

        mask = np.zeros(image_size, dtype=np.uint8)
        for ann in annotations["annotations"]:
            if ann["image_id"] == item["id"]:
                points = np.array(ann["segmentation"], np.int32)
                points = points.reshape((-1, 1, 2))
                mask = cv2.fillPoly(mask, [points], color=(255))

        images.append(image / 255.0)
        masks.append(mask / 255.0)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32).reshape(-1, image_size[0], image_size[1], 1)

X, Y = load_data(image_dir, annotations)

print(f"Loaded {len(X)} images and {len(Y)} masks.")


In [ ]:

# -----------------
# Define Pix2Pix Generator
# -----------------
def build_generator(input_shape=(256, 256, 3)):
    inputs = keras.layers.Input(shape=input_shape)

    # Downsampling
    x = Conv2D(64, (4, 4), strides=2, padding="same", activation=LeakyReLU(0.2))(inputs)
    x = Conv2D(128, (4, 4), strides=2, padding="same")(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(0.2)(x)
    x = Conv2D(256, (4, 4), strides=2, padding="same")(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(0.2)(x)

    # Upsampling
    x = Conv2DTranspose(128, (4, 4), strides=2, padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = Conv2DTranspose(64, (4, 4), strides=2, padding="same", activation="relu")(x)
    x = BatchNormalization()(x)

    outputs = Conv2DTranspose(1, (4, 4), strides=2, padding="same", activation="sigmoid")(x)

    return keras.Model(inputs, outputs)

generator = build_generator()
generator.summary()

# -----------------
# Define Pix2Pix Discriminator
# -----------------
def build_discriminator(input_shape=(256, 256, 3)):
    inputs = keras.layers.Input(shape=input_shape)
    targets = keras.layers.Input(shape=(256, 256, 1))

    x = keras.layers.Concatenate()([inputs, targets])
    x = Conv2D(64, (4, 4), strides=2, padding="same", activation=LeakyReLU(0.2))(x)
    x = Conv2D(128, (4, 4), strides=2, padding="same", activation=LeakyReLU(0.2))(x)
    x = BatchNormalization()(x)
    x = Conv2D(1, (4, 4), strides=1, padding="same", activation="sigmoid")(x)

    return keras.Model([inputs, targets], x)

discriminator = build_discriminator()
discriminator.summary()

# -----------------
# Define Loss & Optimizers
# -----------------
loss_object = keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(disc_generated_output, gen_output, target):
    return loss_object(tf.ones_like(disc_generated_output), disc_generated_output) + \
           tf.reduce_mean(tf.abs(target - gen_output))

def discriminator_loss(disc_real_output, disc_generated_output):
    real_loss = loss_object(tf.ones_like(disc_real_output), disc_real_output)
    fake_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)
    return real_loss + fake_loss

generator_optimizer = keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = keras.optimizers.Adam(2e-4, beta_1=0.5)

# -----------------
# Training Loop (Supports CUDA & CPU)
# -----------------
@tf.function
def train_step(input_image, target):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_output = generator(input_image, training=True)
        disc_real_output = discriminator([input_image, target], training=True)
        disc_generated_output = discriminator([input_image, gen_output], training=True)

        gen_loss = generator_loss(disc_generated_output, gen_output, target)
        disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

# Train the model
EPOCHS = 10
BATCH_SIZE = 2
for epoch in range(EPOCHS):
    for i in range(0, len(X), BATCH_SIZE):
        batch_X = X[i:i+BATCH_SIZE]
        batch_Y = Y[i:i+BATCH_SIZE]

        gen_loss, disc_loss = train_step(batch_X, batch_Y)

    print(f"Epoch {epoch+1}/{EPOCHS} - Gen Loss: {gen_loss:.4f}, Disc Loss: {disc_loss:.4f}")

# -----------------
# Prediction Function
# -----------------
def predict(image):
    image = cv2.resize(image, (256, 256)) / 255.0
    image = np.expand_dims(image, axis=0)

    predicted_mask = generator(image, training=False).numpy()
    predicted_mask = (predicted_mask[0, :, :, 0] > 0.5).astype(np.uint8) * 255

    return predicted_mask

# Test
test_image = X[0]
predicted_mask = predict(test_image)

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.imshow(test_image)
plt.title("Original Image")
plt.subplot(1,2,2)
plt.imshow(predicted_mask, cmap="gray")
plt.title("Predicted Mask")
plt.show()


In [ ]:
!pip install numpy matplotlib opencv-python tensorflow[and-cuda] pillow

import os
import cv2
cv2.utils.logging.setLogLevel(0)  # Suppress OpenCV warnings
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, BatchNormalization, LeakyReLU
from PIL import Image

# -----------------
# Check if CUDA is Available
# -----------------
import torch

# Check if CUDA is available
if torch.cuda.is_available():
    print(f"✅ Running on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected, running on CPU.")

# -----------------
# Define Dataset Paths
# -----------------
image_dir = f"{base_path}/patch/"  # Folder with image tiles
mask_dir = f"{base_path}/patch-binary/"  # Folder with corresponding mask tiles

# -----------------
# Function to Load Dataset
# -----------------
def load_tiled_data(image_dir, mask_dir, image_size=(256, 256)):
    images, masks = [], []

    for image_file in os.listdir(image_dir):
        img_path = os.path.join(image_dir, image_file)
        mask_path = os.path.join(mask_dir, image_file)  # Mask should have the same filename

        if not os.path.exists(mask_path):
            continue  # Skip if corresponding mask is missing

        # Load and preprocess image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, image_size) / 255.0  # Normalize

        # Load and preprocess mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, image_size)
        mask = (mask > 127).astype(np.uint8)  # Convert to binary
        mask = np.expand_dims(mask, axis=-1)  # Add channel dimension

        images.append(image)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

import warnings
# warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter(action="ignore", category=UserWarning)


# Load dataset
X, Y = load_tiled_data(image_dir, mask_dir)
print(f"✅ Loaded {len(X)} images and {len(Y)} masks.")

In [ ]:
def build_generator(input_shape=(256, 256, 3)):
    inputs = tf.keras.layers.Input(shape=input_shape)

    down_stack = [
        tf.keras.layers.Conv2D(64, (4, 4), strides=2, padding="same", activation=tf.keras.layers.LeakyReLU(0.2))(inputs),
        tf.keras.layers.Conv2D(128, (4, 4), strides=2, padding="same", activation=tf.keras.layers.LeakyReLU(0.2))(inputs),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(256, (4, 4), strides=2, padding="same", activation=tf.keras.layers.LeakyReLU(0.2))(inputs),
    ]

    up_stack = [
        tf.keras.layers.Conv2DTranspose(128, (4, 4), strides=2, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2DTranspose(64, (4, 4), strides=2, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
    ]

    x = inputs
    for layer in down_stack:
        x = layer(x)

    for layer in up_stack:
        x = layer(x)

    outputs = tf.keras.layers.Conv2DTranspose(1, (4, 4), strides=1, padding="same", activation="sigmoid")(x)

    return tf.keras.Model(inputs, outputs)

generator = build_generator()
generator.summary()


In [ ]:
def build_discriminator(input_shape=(256, 256, 3)):
    inputs = tf.keras.layers.Input(shape=input_shape)
    targets = tf.keras.layers.Input(shape=(256, 256, 1))

    x = tf.keras.layers.Concatenate()([inputs, targets])
    x = tf.keras.layers.Conv2D(64, (4, 4), strides=2, padding="same", activation=tf.keras.layers.LeakyReLU(0.2))(x)
    x = tf.keras.layers.Conv2D(128, (4, 4), strides=2, padding="same", activation=tf.keras.layers.LeakyReLU(0.2))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Conv2D(1, (4, 4), strides=1, padding="same", activation="sigmoid")(x)

    return tf.keras.Model([inputs, targets], x)


discriminator = build_discriminator()
discriminator.summary()

loss_object = tf.keras.losses.BinaryCrossentropy(from_logits=True)


def generator_loss(disc_generated_output, gen_output, target):
    return loss_object(tf.ones_like(disc_generated_output), disc_generated_output) + \
        tf.reduce_mean(tf.abs(target - gen_output))


def discriminator_loss(disc_real_output, disc_generated_output):
    real_loss = loss_object(tf.ones_like(disc_real_output), disc_real_output)
    fake_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)
    return real_loss + fake_loss


generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)


# Enable CUDA for training
@tf.function
def train_step(input_image, target):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_output = generator(input_image, training=True)
        disc_real_output = discriminator([input_image, target], training=True)
        disc_generated_output = discriminator([input_image, gen_output], training=True)

        gen_loss = generator_loss(disc_generated_output, gen_output, target)
        disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss


# Train the model with CUDA
EPOCHS = 10
BATCH_SIZE = 2  # Adjust based on GPU memory
for epoch in range(EPOCHS):
    for i in range(0, len(X), BATCH_SIZE):
        batch_X = X[i:i + BATCH_SIZE]
        batch_Y = Y[i:i + BATCH_SIZE]

        gen_loss, disc_loss = train_step(batch_X, batch_Y)

    print(f"Epoch {epoch + 1}/{EPOCHS} - Gen Loss: {gen_loss:.4f}, Disc Loss: {disc_loss:.4f}")


def predict(image):
    image = cv2.resize(image, (256, 256)) / 255.0
    image = np.expand_dims(image, axis=0)

    predicted_mask = generator(image, training=False)
    predicted_mask = (predicted_mask[0, :, :, 0] > 0.5).astype(np.uint8) * 255

    return predicted_mask


# Test
test_image = X[0]
predicted_mask = predict(test_image)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(test_image)
plt.title("Original Image")
plt.subplot(1, 2, 2)
plt.imshow(predicted_mask, cmap="gray")
plt.title("Predicted Mask")
plt.show()

In [ ]:
# !pip install  numpy matplotlib opencv-python tensorflow[and-cuda]
# !pip install  numpy matplotlib opencv-python tensorflow
import os
import json
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, BatchNormalization, LeakyReLU
import matplotlib.pyplot as plt

print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))

# Define paths
image_dir = f"{base_path}/Tiles/"  # Update with your dataset path
json_files = [f"{base_path}/Tiles/instances_1_2_3.json", f"{base_path}/Tiles/instances_4_5_6.json",
              f"{base_path}/Tiles/instances_7_8_9.json", f"{base_path}/Tiles/instances_10_11_12.json"]

annotations = {
    "images": [],
    "annotations": []
}

# Load JSON annotations and merge them into a single object
for json_file in json_files:
    with open(json_file) as f:
        data = json.load(f)
        annotations["images"].extend(data["images"])
        annotations["annotations"].extend(data["annotations"])

import cv2
import numpy as np


def load_data(image_dir, annotations, image_size=(256, 256)):
    images, masks = [], []

    for item in annotations["images"]:
        img_path = os.path.join(image_dir, item["file_name"])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, image_size)

        mask = np.zeros(image_size, dtype=np.uint8)
        for ann in annotations["annotations"]:
            if ann["image_id"] == item["id"]:
                points = np.array(ann["segmentation"], np.int32)
                points = points.reshape((-1, 1, 2))
                mask = cv2.fillPoly(mask, [points], color=(255))

        images.append(image / 255.0)
        masks.append(mask / 255.0)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32).reshape(-1, image_size[0],
                                                                                         image_size[1], 1)


X, Y = load_data(image_dir, annotations)

print(f"Loaded {len(X)} images and {len(Y)} masks.")


def build_generator(input_shape=(256, 256, 3)):
    inputs = tf.keras.layers.Input(shape=input_shape)

    # Downsampling
    x = tf.keras.layers.Conv2D(64, (4, 4), strides=2, padding="same", activation=tf.keras.layers.LeakyReLU(0.2))(inputs)
    x = tf.keras.layers.Conv2D(128, (4, 4), strides=2, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Conv2D(256, (4, 4), strides=2, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)

    # Upsampling
    x = tf.keras.layers.Conv2DTranspose(128, (4, 4), strides=2, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Conv2DTranspose(64, (4, 4), strides=2, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)

    outputs = tf.keras.layers.Conv2DTranspose(1, (4, 4), strides=2, padding="same", activation="sigmoid")(x)

    return tf.keras.Model(inputs, outputs)


generator = build_generator()
generator.summary()


def build_discriminator(input_shape=(256, 256, 3)):
    inputs = tf.keras.layers.Input(shape=input_shape)
    targets = tf.keras.layers.Input(shape=(256, 256, 1))

    x = tf.keras.layers.Concatenate()([inputs, targets])
    x = tf.keras.layers.Conv2D(64, (4, 4), strides=2, padding="same", activation=tf.keras.layers.LeakyReLU(0.2))(x)
    x = tf.keras.layers.Conv2D(128, (4, 4), strides=2, padding="same", activation=tf.keras.layers.LeakyReLU(0.2))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Conv2D(1, (4, 4), strides=1, padding="same", activation="sigmoid")(x)

    return tf.keras.Model([inputs, targets], x)


discriminator = build_discriminator()
discriminator.summary()

loss_object = tf.keras.losses.BinaryCrossentropy(from_logits=True)


def generator_loss(disc_generated_output, gen_output, target):
    return loss_object(tf.ones_like(disc_generated_output), disc_generated_output) + \
        tf.reduce_mean(tf.abs(target - gen_output))


def discriminator_loss(disc_real_output, disc_generated_output):
    real_loss = loss_object(tf.ones_like(disc_real_output), disc_real_output)
    fake_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)
    return real_loss + fake_loss


generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)


# Enable CUDA for training
@tf.function
def train_step(input_image, target):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_output = generator(input_image, training=True)
        print(f"Generator output shape: {gen_output.shape}, Expected: (BATCH_SIZE, 256, 256, 1)")
        print(f"Discriminator input shape real: {target.shape}, fake: {gen_output.shape}")
        disc_real_output = discriminator([input_image, target], training=True)
        disc_generated_output = discriminator([input_image, gen_output], training=True)

        gen_loss = generator_loss(disc_generated_output, gen_output, target)
        disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss


# Train the model with CUDA
EPOCHS = 100
BATCH_SIZE = 2  # Adjust based on GPU memory
for epoch in range(EPOCHS):
    for i in range(0, len(X), BATCH_SIZE):
        batch_X = X[i:i + BATCH_SIZE]
        batch_Y = Y[i:i + BATCH_SIZE]

        gen_loss, disc_loss = train_step(batch_X, batch_Y)

    print(f"Epoch {epoch + 1}/{EPOCHS} - Gen Loss: {gen_loss:.4f}, Disc Loss: {disc_loss:.4f}")


def predict(image):
    image = cv2.resize(image, (256, 256)) / 255.0
    image = np.expand_dims(image, axis=0)

    predicted_mask = generator(image, training=False)
    predicted_mask = (predicted_mask[0, :, :, 0].numpy() > 0.5).astype(np.uint8) * 255

    return predicted_mask


# Test
test_image = X[0]
predicted_mask = predict(test_image)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(test_image)
plt.title("Original Image")
plt.subplot(1, 2, 2)
plt.imshow(predicted_mask, cmap="gray")
plt.title("Predicted Mask")
plt.show()

# with tiles and masks


In [ ]:
# -----------------
# Prediction Function
# -----------------
def predict(image):
    image = cv2.resize(image, (256, 256)) / 255.0
    image = np.expand_dims(image, axis=0)
    image = np.transpose(image, (0, 3, 1, 2))  # Change shape to (B, C, H, W)
    image = torch.tensor(image).to(device, dtype=torch.float32)

    with torch.no_grad():
        predicted_mask = generator(image)

    predicted_mask = predicted_mask.cpu().numpy()[0, 0, :, :]
    predicted_mask = (predicted_mask > 0.5).astype(np.uint8) * 255

    return predicted_mask

# Test
test_image = X[101]
predicted_mask = predict(test_image)

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.imshow(test_image)
plt.title("Original Image")
plt.subplot(1,2,2)
plt.imshow(predicted_mask, cmap="gray")
plt.title("Predicted Mask")
plt.show()

In [ ]:

# !pip install torchsummary
# from torchsummary import summary
# -----------------
# Define Pix2Pix Generator in PyTorch
# -----------------
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()

        # Downsampling
        self.down1 = self.conv_block(3, 64)
        self.down2 = self.conv_block(64, 128)
        self.down3 = self.conv_block(128, 256)

        # Upsampling
        self.up1 = self.deconv_block(256, 128)
        self.up2 = self.deconv_block(128, 64)
        self.up3 = nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1)
        self.sigmoid = nn.Sigmoid()


    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2)
        )

    def deconv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

    def forward(self, x):
        x = self.down1(x)
        x = self.down2(x)
        x = self.down3(x)
        x = self.up1(x)
        x = self.up2(x)
        x = self.up3(x)
        return self.sigmoid(x)

# Initialize the generator and move it to the GPU if available
generator = Generator().to(device)
# Print the summary of the generator model
# summary(generator, input_size=(3, 256, 256))

# -----------------
# Define Pix2Pix Discriminator in PyTorch
# -----------------
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        # Modify the input channels to 4 (image + mask)
        self.model = nn.Sequential(
            nn.Conv2d(4, 64, kernel_size=4, stride=2, padding=1),  # Change from 3 to 4 channels
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 1, kernel_size=4, stride=1, padding=1)
        )

    def forward(self, x):
        return self.model(x)

discriminator = Discriminator().to(device)
# summary(discriminator, input_size=(3, 256, 256))


# -----------------
# Define Loss & Optimizers
# -----------------
loss_fn = nn.BCEWithLogitsLoss()

# Optimizers
generator_optimizer = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
discriminator_optimizer = optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))

# -----------------
# Training Loop (Supports CUDA & CPU)
# -----------------
def train_step(input_image, target):
    # Convert images from (batch_size, height, width, channels) to (batch_size, channels, height, width)
    input_image = torch.tensor(input_image).to(device).permute(0, 3, 1, 2)  # Change shape to (B, C, H, W)
    target = torch.tensor(target).to(device).permute(0, 3, 1, 2)  # Change shape to (B, C, H, W)

    # Forward pass through generator and discriminator
    gen_output = generator(input_image)

    disc_real_output = discriminator(torch.cat((input_image, target), dim=1))
    disc_generated_output = discriminator(torch.cat((input_image, gen_output), dim=1))

    # Losses
    gen_loss = loss_fn(disc_generated_output, torch.ones_like(disc_generated_output)) + torch.mean(torch.abs(target - gen_output))
    disc_loss = (loss_fn(disc_real_output, torch.ones_like(disc_real_output)) +
                 loss_fn(disc_generated_output, torch.zeros_like(disc_generated_output)))

    # Backpropagation
    generator_optimizer.zero_grad()
    discriminator_optimizer.zero_grad()

    # Backprop through generator first
    gen_loss.backward(retain_graph=True)  # Retain the graph for the next backward pass

    # Backprop through discriminator
    disc_loss.backward()

    # Update the weights
    generator_optimizer.step()
    discriminator_optimizer.step()

    return gen_loss.item(), disc_loss.item()





EPOCHS = 60
BATCH_SIZE = 128

for epoch in range(EPOCHS):
    # Create a tqdm progress bar for each epoch, based on the number of batches
    progress_bar = tqdm(range(0, len(X), BATCH_SIZE), desc=f"Epoch {epoch+1}/{EPOCHS}", ncols=100)

    for i in progress_bar:
        batch_X = X[i:i+BATCH_SIZE]
        batch_Y = Y[i:i+BATCH_SIZE]

        gen_loss, disc_loss = train_step(batch_X, batch_Y)

        # Update the progress bar with the generator and discriminator losses
        progress_bar.set_postfix(Gen_Loss=gen_loss, Disc_Loss=disc_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} - Gen Loss: {gen_loss:.4f}, Disc Loss: {disc_loss:.4f}")

